# RoBERTa detector (role B)
No logic here: every cell just calls `src/train_roberta.py`. Settings: GPU T4, Internet on, dataset `team20-m4-processed` attached.
Run order follows issue #10: `indist` first (keep its model), then LOGO, then paraphrase, then cross-domain. Commit/download results as you go; stop if GPU quota runs out.

**What this step does:** downloads the project code from GitHub, installs the Python packages, links the attached dataset into `data/processed`, and checks that every id in the split files exists in the data. It is needed because a fresh Kaggle session starts empty.

**What you should see:** the `ls` listing shows `m4_en.parquet`, and the split check finishes without an error.

**How long:** quick, but it depends on the network (clone and pip install need Internet). No measured time is recorded for this step.

In [ ]:
!git clone -b main https://github.com/NIRMAL20062/generated_text_detection.git
%cd generated_text_detection
# If pip breaks torch, comment this line out: Kaggle's preinstalled packages are enough.
!pip install -q -r requirements.txt
# The dataset mounts under /kaggle/input/... (path differs by account), so find it.
!mkdir -p data && ln -sfn "$(dirname "$(find /kaggle/input -name m4_en.parquet | head -1)")" data/processed
!ls -l data/processed/   # must show m4_en.parquet
!python -m src.data --check-splits

## 1. `indist`: train, then score the original paraphrase test set with the same model

**What this step does:** trains the RoBERTa detector on the in-distribution split and scores its test set. The trained model is saved in `outputs/roberta/indist/model` because later steps reuse it.

**What you should see:** training progress, then `results/roberta/indist/` with predictions and `run_info.json`.

**How long:** about 9 minutes on a Kaggle T4 GPU (training 462.4 s, whole run 516.5 s, from `results/roberta/indist/run_info.json`).

In [ ]:
!python -m src.train_roberta --setting indist

**What this step does:** does not train. It loads the saved `indist` model and scores the original (unparaphrased) paraphrase test texts, so the paraphrase results can be compared against the same texts before the attack.

**What you should see:** predictions in `results/roberta/paraphrase_orig/`.

**How long:** about 20 seconds on a Kaggle T4 GPU (20.6 s, from `results/roberta/paraphrase_orig/run_info.json`).

In [ ]:
!python -m src.train_roberta --setting paraphrase_orig --predict-only --model outputs/roberta/indist/model

## 2. Leave-one-generator-out (most important for the research question)
`--no-save` keeps disk free: only the `indist` model is needed later.

**What this step does:** loops over 5 generators (chatGPT, davinci, cohere, dolly, bloomz). For each one it trains RoBERTa on the other four and tests on the held-out one. This shows whether the detector works on a generator it never saw. The model is not saved (`--no-save`).

**What you should see:** 5 separate training runs, each writing `results/roberta/logo_<generator>/`.

**How long:** 5 runs of about 6 minutes each on a Kaggle T4 GPU (348 to 354 s per run, from the `results/roberta/logo_*/run_info.json` files), so about 30 minutes in total.

In [ ]:
for gen in ["chatGPT", "davinci", "cohere", "dolly", "bloomz"]:
    !python -m src.train_roberta --setting logo_{gen} --no-save

## 3. Paraphrase attack (only after Vishal's issue #13 lands `splits/paraphrase*.json` and `paraphrased.parquet`)

**What this step does:** loops over the 2 settings `paraphrase` and `paraphrase_human`. For each it scores the paraphrased texts with the saved `indist` model (no training). A setting is skipped with a message if its split file does not exist yet, so rerun this cell after the paraphrase step.

**What you should see:** predictions in `results/roberta/paraphrase/` and `results/roberta/paraphrase_human/`, or a "skip" message.

**How long:** 2 runs of about 20 seconds each on a Kaggle T4 GPU (21.3 s and 20.9 s, from `results/roberta/paraphrase*/run_info.json`).

In [ ]:
import os
for s in ["paraphrase", "paraphrase_human"]:
    if os.path.exists(f"splits/{s}.json"):
        !python -m src.train_roberta --setting {s} --predict-only --model outputs/roberta/indist/model
    else:
        print(f"splits/{s}.json not on this branch yet: skip, rerun this cell after #13 merges")

## 4. Cross-domain

**What this step does:** loops over 5 text domains (wikipedia, wikihow, reddit, arxiv, peerread). For each one it trains RoBERTa on the other domains and tests on the held-out one. This shows whether the detector works on a domain it never saw. The model is not saved (`--no-save`).

**What you should see:** 5 training runs, each writing `results/roberta/xdomain_<domain>/`.

**How long:** 5 runs of about 8 minutes each on a Kaggle T4 GPU (469 to 476 s per run, from the `results/roberta/xdomain_*/run_info.json` files), so about 40 minutes in total.

In [ ]:
for dom in ["wikipedia", "wikihow", "reddit", "arxiv", "peerread"]:
    !python -m src.train_roberta --setting xdomain_{dom} --no-save

## 5. Metrics with the shared evaluator

**What this step does:** computes the metrics (for example AUROC) for every `preds.csv` with the shared evaluator `src/eval.py`, and prints one summary line per setting. If `src/eval.py` is missing on this branch, it downloads just that file first.

**What you should see:** one summary line per setting, and no line starting with `FAILED`.

**How long:** quick (no measured time; it only reads the saved prediction files, no GPU is needed).

In [ ]:
# src/eval.py is only on the a-data branch until PR #19 merges, so fetch just that file if it is missing.
import glob, os, subprocess, sys, urllib.request
EVAL_URL = "https://raw.githubusercontent.com/NIRMAL20062/generated_text_detection/a-data/src/eval.py"
if not os.path.exists("src/eval.py"):
    urllib.request.urlretrieve(EVAL_URL, "src/eval.py")
for p in sorted(glob.glob("results/roberta/*/preds.csv")):
    r = subprocess.run([sys.executable, "-m", "src.eval", p], capture_output=True, text=True)
    print((r.stdout.strip().splitlines() or [""])[0] if r.returncode == 0 else f"FAILED {p}: {r.stderr.strip().splitlines()[-1]}")

**What this step does:** lists the RoBERTa result folders. Download this folder from the Output tab, then commit it from your own machine.

**What you should see:** 14 folders (`indist`, 5 `logo_*`, `paraphrase_orig`, `paraphrase`, `paraphrase_human`, 5 `xdomain_*`) if everything ran.

**How long:** instant.

In [ ]:
!ls results/roberta   # download this folder from the Output tab, then commit it from your own machine